In [1]:
%load_ext line_profiler

In [2]:
%load_ext autoreload

In [58]:
%autoreload
from functools import cache, wraps
import inspect

import wrapt
from frozendict import frozendict

from microscope_calibration.common.model import symbol_maker, lambdify as mc_lambdify
import sympy as sym
import jax
import jax_dataclasses as jdc

In [4]:
@jdc.pytree_dataclass
class Ray:
    y: float
    x: float

@jdc.pytree_dataclass
class Params:
    a: float
    b: float

    @classmethod
    @cache
    def _mk_trace(cls, modules):
        def _trace_impl(arg) -> Ray:
            self, ray = arg
            return Ray(
                y=sym.Heaviside(self.a) * ray.y,
                x=ray.x + self.b,
            )

        p = symbol_maker(cls)
        r = symbol_maker(Ray)
        
        outer = mc_lambdify((p, r), _trace_impl((p, r)), modules=modules)
        return outer

    def trace(self, ray: Ray, modules=sym) -> Ray:
        f = self._mk_trace(modules=modules)
        res = f((self, ray))
        return res

In [5]:
%autoreload
p = symbol_maker(Params, postfix="param")
r = symbol_maker(Ray, postfix="ray")

In [6]:
%time p.trace(r)

CPU times: user 27.1 ms, sys: 158 μs, total: 27.3 ms
Wall time: 26.9 ms


Ray(y=y_ray_0*Heaviside(a_param_0, 0.5), x=b_param_1 + x_ray_1)

In [7]:
%time p.trace(r)

CPU times: user 113 μs, sys: 0 ns, total: 113 μs
Wall time: 117 μs


Ray(y=y_ray_0*Heaviside(a_param_0, 0.5), x=b_param_1 + x_ray_1)

In [8]:
pp = Params(a=0, b=1)
rr = Ray(y=jax.numpy.array((0, 1, 2)), x=jax.numpy.array((3, 2, 1)))

In [9]:
%time pp.trace(rr, modules=jax.numpy)

CPU times: user 47 ms, sys: 3.71 ms, total: 50.7 ms
Wall time: 49.6 ms


Ray(y=Array([0. , 0.5, 1. ], dtype=float64, weak_type=True), x=Array([4, 3, 2], dtype=int64))

In [10]:
%time pp.trace(rr, modules=jax.numpy)

CPU times: user 452 μs, sys: 25 μs, total: 477 μs
Wall time: 318 μs


Ray(y=Array([0. , 0.5, 1. ], dtype=float64, weak_type=True), x=Array([4, 3, 2], dtype=int64))

In [11]:
def deco_me(a: Ray, b: int=23):
    pass

In [12]:
# inspect.get_annotations(deco_me)
inspect.getfullargspec(deco_me)

FullArgSpec(args=['a', 'b'], varargs=None, varkw=None, defaults=(23,), kwonlyargs=[], kwonlydefaults=None, annotations={'a': <class '__main__.Ray'>, 'b': <class 'int'>})

In [67]:
def normalized_args(wrapped, args, kwargs):
    sig = inspect.signature(wrapped)
    bound = sig.bind(*args, **kwargs)
    bound.apply_defaults()
    all_args = bound.arguments
    return all_args


@cache
def actual_lambdify(wrapped, modules, sample_args, sample_kwargs, sym_kwargs):
    sig = inspect.signature(wrapped)
    spec = inspect.getfullargspec(wrapped)
    partial_bound = sig.bind_partial(*sample_args, **sample_kwargs)
    # partial_bound.apply_defaults()
    generated_args = {}
    for arg, cls in spec.annotations.items():
        if arg not in partial_bound.arguments:
            generated_args[arg] = symbol_maker(cls)
    partial_bound.arguments.update(generated_args)
    normalized = normalized_args(wrapped, args=tuple(), kwargs=partial_bound.arguments)

    f = mc_lambdify(normalized, wrapped(**normalized), modules=modules, **sym_kwargs)
    return f


def lambdify(recurse_for=tuple(), modules=sym, args=None, kwargs=None, **sym_kwargs):
    # Rename to keep outer API succinct
    sample_args = tuple() if args is None else args 
    sample_kwargs = {} if kwargs is None else kwargs
    
    @wrapt.decorator
    def lambdified(wrapped, instance, args, kwargs):
        n_args = normalized_args(wrapped, args, kwargs)
        f = actual_lambdify(
            wrapped=wrapped,
            modules=modules,
            sample_args=frozendict(sample_args),
            sample_kwargs=frozendict(sample_kwargs),
            sym_kwargs=frozendict(sym_kwargs),
        )
        return f(n_args)

    return lambdified

In [68]:
@lambdify(modules=jax.numpy)
def deco_me(a: Ray, b: int=23):
    print("deco_me", a, b)
    return (a, b)

In [70]:
%time deco_me(Ray(y=1, x=1))

CPU times: user 151 μs, sys: 0 ns, total: 151 μs
Wall time: 159 μs


(Ray(y=1, x=1), 23)

In [64]:
sig = inspect.signature(deco_me)
spec = inspect.getfullargspec(deco_me)

In [17]:
partial_bound = sig.bind_partial(b=42)

In [18]:
partial_bound.apply_defaults()

In [19]:
partial_bound.arguments

{'b': 42}

In [20]:
%autoreload
symbol_maker(Ray)

Ray(y=y_0, x=x_1)

In [21]:
@jdc.pytree_dataclass
class BadRay:
    y: float
    x: str

In [22]:
symbol_maker(BadRay)

TypeError: Can't generate symbol for type <class 'str'>.

In [ ]:
generated_args = {}

for arg, cls in spec.annotations.items():
    if arg not in partial_bound.arguments:
        generated_args[arg] = symbol_maker(cls)
generated_args

In [ ]:
sig.parameters

In [ ]:
from numbers import Number

In [ ]:
issubclass(int, Number)

In [ ]:
int.__name__